# Q6 Appendix: All-metric raw means in tidy/long format (Accuracy / Precision / Recall / Macro-F1)

This appendix reports the Q6 comparisons — the three-way model-family comparison (**Deep Learning / Classical / Ensemble**) and the two case studies — with all four evaluation metrics (Accuracy, Precision, Recall, and Macro-F1 = `f1_score`) as raw mean-over-runs values.

The tables are **tidy/long**: exactly one column per metric (Accuracy, Precision, Recall, Macro-F1, in that order), and the condition that was previously widened across the columns (model family, evaluation regime, case-study stage, deployment target) becomes a single label column on the left. All headline derived columns (family deltas, split-drop, transfer-drop, temporal_drop, total_drop, deploy_drop) and all statistical machinery (Mann-Whitney p, Holm p, Cohen's d, significance, checkmarks) are dropped. The Macro-F1 values are identical to the prior wide tables.

**Q6 invariants preserved:** drop split "Random with same distribution"; `enable_sequences == True` only (sequence-based); families `DL={gru, lstm, rnn, mlp, nn}`, `Classical={knn, logistic-regression, svm}`, `Ensemble={random-forest, lightgbm, xgboost}`; the SetA$\leftrightarrow$SetC pair is excluded from the cross-network regime (same physical network, different capture period); `eval_type` = In-network vs Cross-network; average over `run_no`. The setup melts **all four metrics** into a `metric` column and carries them through.


## Setup, load, melt, and run-averaging (all four metrics)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths (notebook runs from A/) ────────────────────────────────────────────
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

NETWORKS = ['SetA', 'SetB', 'SetC', 'SetD']
PRIMARY  = 'f1_score'
# All four metrics carried through (NOT filtered to f1_score)
METRIC_ORDER = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}

# ── Three model families (verbatim from Q6) ──────────────────────────────────
DL_MODELS        = ['gru', 'lstm', 'rnn', 'mlp', 'nn']
CLASSICAL_MODELS = ['knn', 'logistic-regression', 'svm']
ENSEMBLE_MODELS  = ['random-forest', 'lightgbm', 'xgboost']
ALL_MODELS       = DL_MODELS + CLASSICAL_MODELS + ENSEMBLE_MODELS
FAMILIES         = ['Deep Learning', 'Classical', 'Ensemble']

def family_of(m):
    if m in DL_MODELS:        return 'Deep Learning'
    if m in CLASSICAL_MODELS: return 'Classical'
    return 'Ensemble'

MODEL_TEX = {'gru': 'GRU', 'lstm': 'LSTM', 'rnn': 'RNN', 'mlp': 'MLP', 'nn': 'NN',
             'knn': 'k-NN', 'logistic-regression': 'Logistic Regression', 'svm': 'SVM',
             'random-forest': 'Random Forest', 'lightgbm': 'LightGBM',
             'xgboost': 'XGBoost'}

# SetA & SetC are the same physical network at different capture times; exclude that
# pair from cross-network transfer (studied separately in Case Study A).
EXCLUDE_PAIRS = {('SetA', 'SetC'), ('SetC', 'SetA')}

# ── Load raw data (Q6 filters, verbatim) ─────────────────────────────────────
raw = pd.read_csv(DATA_PATH, index_col=0)
raw = raw[raw['split'] != 'Random with same distribution']
raw = raw[raw['enable_sequences'] == True]          # sequence-based approach only

# ── Melt ALL metrics into long form (carry every metric, not just f1) ────────
metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']
long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value')
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')

long['family']    = long['model'].map(family_of)
long['eval_type'] = np.where(long['train_set'] == long['test_set'],
                             'In-network', 'Cross-network')

# Keep all four metrics; drop only NaN values. This mirrors Q6's `f1` frame but
# retains every metric in the `metric` dimension.
allm = long[long['metric'].isin(METRIC_ORDER)].dropna(subset=['value']).copy()

print('Sequence-based rows (all metrics):', f'{len(allm):,}')
print('Metrics present:', sorted(allm['metric'].unique()))
print('Models present per family:')
for fam in FAMILIES:
    ms = [m for m in ALL_MODELS if family_of(m) == fam and m in allm['model'].unique()]
    print(f'  {fam:<14}: {ms}')


Sequence-based rows (all metrics): 26,432
Metrics present: ['accuracy', 'f1_score', 'precision', 'recall']
Models present per family:
  Deep Learning : ['gru', 'lstm', 'rnn', 'mlp', 'nn']
  Classical     : ['knn', 'logistic-regression', 'svm']
  Ensemble      : ['random-forest', 'lightgbm', 'xgboost']


The next cell averages every metric over training runs for each unique configuration (the `metric` dimension is preserved), then splits into in-network and cross-network frames, excluding the SetA$\leftrightarrow$SetC pair from cross-network — exactly as Q6 does, but with all four metrics retained.


In [2]:
group_keys = ['model', 'family', 'task', 'split', 'train_set', 'test_set',
              'eval_type', 'metric']
avg = (allm.groupby(group_keys, as_index=False)
           .agg(mean_value=('value', 'mean'), n_runs=('value', 'count')))

avg_in    = avg[avg['eval_type'] == 'In-network'].copy()
avg_cross = avg[avg['eval_type'] == 'Cross-network'].copy()
avg_cross = avg_cross[~avg_cross.apply(
    lambda r: (r['train_set'], r['test_set']) in EXCLUDE_PAIRS, axis=1)]

print('Averaged rows (all):          ', len(avg))
print('Averaged rows (in-network):   ', len(avg_in))
print('Averaged rows (cross-network):', len(avg_cross))
avg.head()


Averaged rows (all):           2816
Averaged rows (in-network):    704
Averaged rows (cross-network): 1760


,model,family,task,split,train_set,test_set,eval_type,metric,mean_value,n_runs
0,gru,Deep Learning,Binary,Random split,SetA,SetA,In-network,accuracy,0.941879,10
1,gru,Deep Learning,Binary,Random split,SetA,SetA,In-network,f1_score,0.801669,10
2,gru,Deep Learning,Binary,Random split,SetA,SetA,In-network,precision,0.801336,10
3,gru,Deep Learning,Binary,Random split,SetA,SetA,In-network,recall,0.807207,10
4,gru,Deep Learning,Binary,Random split,SetA,SetB,Cross-network,accuracy,0.935300,10


The helper below returns one tidy row: the mean of each of the four metrics (Accuracy, Precision, Recall, Macro-F1, in that order) over a subset, optionally filtered to a single group value. Every table below builds its rows from this helper, so each row is one condition and there are exactly four metric columns.


In [3]:
def metric_row(sub, group_col=None, group_val=None):
    """Mean of each metric over `sub` (optionally filtered to group_col == group_val).

    Returns an ordered dict {MetricLabel: mean_value} for the four metrics in
    METRIC_ORDER (Accuracy, Precision, Recall, Macro-F1). NaN if empty. This is the
    tidy per-row building block: one row = one condition, four metric columns.
    """
    s = sub if group_col is None else sub[sub[group_col] == group_val]
    out = {}
    for m in METRIC_ORDER:
        out[METRIC_LABEL[m]] = s[s['metric'] == m]['mean_value'].mean()
    return out

def f1_mean(sub, mask=None):
    """Mean Macro-F1 (f1_score) over `sub`, optionally further filtered by `mask`.

    Used only to derive the model sort order (overall in-network Macro-F1); it is not
    emitted as a table column.
    """
    s = sub if mask is None else sub[mask]
    return s[s['metric'] == PRIMARY]['mean_value'].mean()

# Fixed metric-column order for every tidy table.
METRIC_COLS = [METRIC_LABEL[m] for m in METRIC_ORDER]  # Accuracy, Precision, Recall, Macro-F1


## q6_appendix_overall

Overall three-way family comparison, tidy/long. Grouping columns `[Eval regime, Task, Split]`; the model family (Deep Learning / Classical / Ensemble, fixed order) is a single `Family` label column, giving 3 family rows per (regime, task, split). Metric columns: Accuracy, Precision, Recall, Macro-F1. Regime/task/split iteration order matches the article; SetA$\leftrightarrow$SetC excluded from the cross-network regime.


In [4]:
records = []
for eval_type, sub in [('In-network', avg_in), ('Cross-network', avg_cross)]:
    for task in ['Binary', 'Multiclass']:
        for split in ['Random split', 'Time split']:
            base = sub[(sub['task'] == task) & (sub['split'] == split)]
            for fam in FAMILIES:  # fixed order: DL, Classical, Ensemble
                row = {'Eval regime': eval_type, 'Task': task, 'Split': split,
                       'Family': fam}
                row.update(metric_row(base, 'family', fam))
                records.append(row)

idx_cols = ['Eval regime', 'Task', 'Split', 'Family']
t_overall = pd.DataFrame(records)[idx_cols + METRIC_COLS]
t_overall[METRIC_COLS] = t_overall[METRIC_COLS].round(4)

t_overall.to_csv(TAB_DIR / 'q6_appendix_overall.csv', index=False)
caption = ('Q6 appendix: means over runs of Accuracy, Precision, Recall, and Macro-F1 '
           'by model family (Deep Learning, Classical, Ensemble), sequence-based '
           'approach, by evaluation regime, task, and split; SetA$\\leftrightarrow$SetC '
           'excluded from the cross-network regime.')
latex = t_overall.to_latex(index=False, float_format='%.4f', escape=False,
                           caption=caption, label='tab:q6_appendix_overall')
(TAB_DIR / 'q6_appendix_overall.tex').write_text(latex)
print('Saved q6_appendix_overall  shape:', t_overall.shape)
t_overall


Saved q6_appendix_overall  shape: (24, 8)


,Eval regime,Task,Split,Family,Accuracy,Precision,Recall,Macro-F1
0,In-network,Binary,Random split,Deep Learning,0.9701,0.9138,0.8828,0.8852
1,In-network,Binary,Random split,Classical,0.9522,0.8446,0.8241,0.8206
2,In-network,Binary,Random split,Ensemble,0.9875,0.9627,0.9330,0.9470
3,In-network,Binary,Time split,Deep Learning,0.9769,0.8971,0.8091,0.8331
4,In-network,Binary,Time split,Classical,0.9716,0.8639,0.7621,0.7836
5,In-network,Binary,Time split,Ensemble,0.9758,0.8863,0.8485,0.8536
6,In-network,Multiclass,Random split,Deep Learning,0.9740,0.8939,0.7551,0.7948
7,In-network,Multiclass,Random split,Classical,0.9591,0.8016,0.6240,0.6618
8,In-network,Multiclass,Random split,Ensemble,0.9864,0.9412,0.8668,0.8969
9,In-network,Multiclass,Time split,Deep Learning,0.9737,0.8437,0.6921,0.7235


## q6_appendix_per_network

Per-network family comparison (in-network), tidy/long. Grouping columns `[Network, Task]`; the model family (Deep Learning / Classical / Ensemble, fixed order) is a single `Family` label column. Metric columns: Accuracy, Precision, Recall, Macro-F1 (raw family means averaged over splits and over the models of each family).


In [5]:
records = []
for net in NETWORKS:
    for task in ['Binary', 'Multiclass']:
        base = avg_in[(avg_in['test_set'] == net) & (avg_in['task'] == task)]
        for fam in FAMILIES:  # fixed order: DL, Classical, Ensemble
            row = {'Network': net, 'Task': task, 'Family': fam}
            row.update(metric_row(base, 'family', fam))
            records.append(row)

idx_cols = ['Network', 'Task', 'Family']
t_net = pd.DataFrame(records)[idx_cols + METRIC_COLS]
t_net[METRIC_COLS] = t_net[METRIC_COLS].round(4)

t_net.to_csv(TAB_DIR / 'q6_appendix_per_network.csv', index=False)
caption = ('Q6 appendix: means over runs of Accuracy, Precision, Recall, and Macro-F1 '
           'by model family (in-network evaluation, sequence-based approach), per '
           'network and task, averaged over the models of each family and over splits; '
           'SetA$\\leftrightarrow$SetC excluded from the cross-network regime.')
latex = t_net.to_latex(index=False, float_format='%.4f', escape=False,
                       caption=caption, label='tab:q6_appendix_per_network')
(TAB_DIR / 'q6_appendix_per_network.tex').write_text(latex)
print('Saved q6_appendix_per_network  shape:', t_net.shape)
t_net


Saved q6_appendix_per_network  shape: (24, 7)


,Network,Task,Family,Accuracy,Precision,Recall,Macro-F1
0,SetA,Binary,Deep Learning,0.9235,0.7166,0.6294,0.6197
1,SetA,Binary,Classical,0.9110,0.6545,0.5618,0.5581
2,SetA,Binary,Ensemble,0.9424,0.7517,0.7197,0.7104
3,SetA,Multiclass,Deep Learning,0.9332,0.6710,0.4822,0.5151
4,SetA,Multiclass,Classical,0.9336,0.5821,0.4091,0.4312
5,SetA,Multiclass,Ensemble,0.9406,0.7395,0.6274,0.6588
6,SetB,Binary,Deep Learning,0.9928,0.9718,0.9193,0.9427
7,SetB,Binary,Classical,0.9859,0.9017,0.8965,0.8897
8,SetB,Binary,Ensemble,0.9955,0.9845,0.9444,0.9634
9,SetB,Multiclass,Deep Learning,0.9927,0.9561,0.8338,0.8782


## q6_appendix_per_model

Per-model comparison, tidy/long. Grouping columns `[Model, Family]`; the evaluation regime (In-network (Random split) / In-network (Time split) / Cross-network, fixed order) is a single `Regime` label column, giving 3 regime rows per model. Metric columns: Accuracy, Precision, Recall, Macro-F1. Models are sorted by overall in-network Macro-F1 descending, then by regime in fixed order. Cross-network averages all admissible transfers (SetA$\leftrightarrow$SetC excluded).


In [6]:
REGIME_ORDER = ['In-network (Random split)', 'In-network (Time split)', 'Cross-network']

records = []
for model in ALL_MODELS:
    if model not in avg_in['model'].unique():
        continue
    fam         = family_of(model)
    in_model    = avg_in[avg_in['model'] == model]
    in_rand     = in_model[in_model['split'] == 'Random split']
    in_time     = in_model[in_model['split'] == 'Time split']
    cross_model = avg_cross[avg_cross['model'] == model]
    sort_key = f1_mean(in_model)  # overall in-network Macro-F1, for model ordering
    regime_subs = {
        'In-network (Random split)': in_rand,
        'In-network (Time split)':   in_time,
        'Cross-network':             cross_model,
    }
    for regime in REGIME_ORDER:  # fixed regime order per model
        row = {'Model': MODEL_TEX[model], 'Family': fam, 'Regime': regime,
               '_sort': sort_key}
        row.update(metric_row(regime_subs[regime]))
        records.append(row)

idx_cols = ['Model', 'Family', 'Regime']
t_model = pd.DataFrame(records)
# Sort models by in-network overall Macro-F1 DESC, then regime in fixed order.
t_model['_regrank'] = t_model['Regime'].map({r: i for i, r in enumerate(REGIME_ORDER)})
t_model = (t_model.sort_values(['_sort', '_regrank'], ascending=[False, True])
                  .drop(columns=['_sort', '_regrank'])
                  .reset_index(drop=True))
t_model = t_model[idx_cols + METRIC_COLS]
t_model[METRIC_COLS] = t_model[METRIC_COLS].round(4)

t_model.to_csv(TAB_DIR / 'q6_appendix_per_model.csv', index=False)
caption = ('Q6 appendix: per-model means over runs of Accuracy, Precision, Recall, and '
           'Macro-F1 (sequence-based) across three evaluation regimes '
           '(In-network under the random split, In-network under the time split, and '
           'Cross-network over all admissible transfers, SetA$\\leftrightarrow$SetC '
           'excluded), with models sorted by overall in-network Macro-F1.')
latex = t_model.to_latex(index=False, float_format='%.4f', escape=False,
                         caption=caption, label='tab:q6_appendix_per_model')
(TAB_DIR / 'q6_appendix_per_model.tex').write_text(latex)
print('Saved q6_appendix_per_model  shape:', t_model.shape)
t_model


Saved q6_appendix_per_model  shape: (33, 7)


,Model,Family,Regime,Accuracy,Precision,Recall,Macro-F1
0,LightGBM,Ensemble,In-network (Random split),0.9873,0.9520,0.9020,0.9233
1,LightGBM,Ensemble,In-network (Time split),0.9692,0.8447,0.8171,0.8202
2,LightGBM,Ensemble,Cross-network,0.8399,0.6972,0.6660,0.6131
3,XGBoost,Ensemble,In-network (Random split),0.9869,0.9505,0.8999,0.9214
4,XGBoost,Ensemble,In-network (Time split),0.9766,0.8860,0.7991,0.8183
5,XGBoost,Ensemble,Cross-network,0.8330,0.6967,0.6470,0.5993
6,Random Forest,Ensemble,In-network (Random split),0.9867,0.9534,0.8980,0.9212
7,Random Forest,Ensemble,In-network (Time split),0.9789,0.8740,0.7882,0.8089
8,Random Forest,Ensemble,Cross-network,0.8342,0.7084,0.6661,0.6087
9,GRU,Deep Learning,In-network (Random split),0.9802,0.9164,0.8657,0.8843


## q6_appendix_cs_temporal

Case Study A — temporal degradation over a $\approx$4-month gap on the same physical network (SetA $\to$ SetC) — tidy/long. Grouping column `[Family]`; the temporal-ladder stage (SetA$\to$SetA (Random split) / SetA$\to$SetA (Time split) / SetA$\to$SetC (Time split), fixed order) is a single `Stage` label column, giving 3 stage rows per family. Metric columns: Accuracy, Precision, Recall, Macro-F1.


In [7]:
# Temporal frame: train on SetA, evaluate on SetA or SetC (all metrics kept).
tempo = allm[(allm['train_set'] == 'SetA') &
             (allm['test_set'].isin(['SetA', 'SetC']))].copy()
tempo_avg = (tempo.groupby(['model', 'family', 'task', 'split', 'test_set', 'metric'],
                           as_index=False)['value'].mean()
                  .rename(columns={'value': 'mean_value'}))

def _rung_row(fam, split, test_set):
    """Tidy four-metric row for one family at one temporal-ladder rung."""
    sub = tempo_avg[(tempo_avg['family'] == fam) & (tempo_avg['split'] == split) &
                    (tempo_avg['test_set'] == test_set)]
    return metric_row(sub)  # uses mean_value / metric columns of `sub`

# Fixed stage order: SetA->SetA random, SetA->SetA time, SetA->SetC time.
STAGE_ORDER = ['SetA→SetA (Random split)', 'SetA→SetA (Time split)',
               'SetA→SetC (Time split)']
STAGE_SPEC = {
    'SetA→SetA (Random split)': ('Random split', 'SetA'),
    'SetA→SetA (Time split)':   ('Time split',   'SetA'),
    'SetA→SetC (Time split)':   ('Time split',   'SetC'),
}

records = []
for fam in FAMILIES:
    for stage in STAGE_ORDER:
        split, test_set = STAGE_SPEC[stage]
        row = {'Family': fam, 'Stage': stage}
        row.update(_rung_row(fam, split, test_set))
        records.append(row)

idx_cols = ['Family', 'Stage']
t_cs_temporal = pd.DataFrame(records)[idx_cols + METRIC_COLS]
t_cs_temporal[METRIC_COLS] = t_cs_temporal[METRIC_COLS].round(4)

t_cs_temporal.to_csv(TAB_DIR / 'q6_appendix_cs_temporal.csv', index=False)
caption = ('Q6 appendix, Case Study A: means over runs of Accuracy, Precision, Recall, '
           'and Macro-F1 by model family (sequence-based) along the temporal ladder '
           'SetA$\\to$SetA (random split), SetA$\\to$SetA (time split), and '
           'SetA$\\to$SetC (time split, same physical network $\\approx$4 months later). '
           'SetA$\\leftrightarrow$SetC is studied here as a temporal gap rather than as '
           'a cross-network transfer.')
latex = t_cs_temporal.to_latex(index=False, float_format='%.4f', escape=False,
                               caption=caption, label='tab:q6_appendix_cs_temporal')
(TAB_DIR / 'q6_appendix_cs_temporal.tex').write_text(latex)
print('Saved q6_appendix_cs_temporal  shape:', t_cs_temporal.shape)
t_cs_temporal


Saved q6_appendix_cs_temporal  shape: (9, 6)


,Family,Stage,Accuracy,Precision,Recall,Macro-F1
0,Deep Learning,SetA→SetA (Random split),0.9204,0.7584,0.6453,0.6541
1,Deep Learning,SetA→SetA (Time split),0.9363,0.6291,0.4663,0.4807
2,Deep Learning,SetA→SetC (Time split),0.9347,0.6007,0.4725,0.4654
3,Classical,SetA→SetA (Random split),0.8949,0.6317,0.5208,0.5292
4,Classical,SetA→SetA (Time split),0.9496,0.6049,0.4501,0.4602
5,Classical,SetA→SetC (Time split),0.7899,0.5863,0.4332,0.3785
6,Ensemble,SetA→SetA (Random split),0.9622,0.8722,0.8160,0.8409
7,Ensemble,SetA→SetA (Time split),0.9208,0.6190,0.5311,0.5283
8,Ensemble,SetA→SetC (Time split),0.9236,0.5346,0.5229,0.4880


## q6_appendix_cs_deploy

Case Study B — cross-network deployment from the strongest source SetC (time split) — tidy/long. Grouping column `[Family]`; the deployment target (SetC (in-network) / SetC$\to$SetB / SetC$\to$SetD, fixed order) is a single `Target` label column, giving 3 target rows per family. Metric columns: Accuracy, Precision, Recall, Macro-F1. SetA is excluded as a target (same physical network as SetC).


In [8]:
# Deploy frame: train on SetC, evaluate on SetC/SetB/SetD, time split (all metrics).
dep = allm[(allm['train_set'] == 'SetC') &
           (allm['test_set'].isin(['SetC', 'SetB', 'SetD'])) &
           (allm['split'] == 'Time split')].copy()
dep_avg = (dep.groupby(['model', 'family', 'test_set', 'metric'], as_index=False)['value']
              .mean().rename(columns={'value': 'mean_value'}))

def _dep_row(fam, test_set):
    """Tidy four-metric row for one family at one deployment target."""
    sub = dep_avg[(dep_avg['family'] == fam) & (dep_avg['test_set'] == test_set)]
    return metric_row(sub)

# Fixed target order: SetC in-network, SetC->SetB, SetC->SetD.
TARGET_ORDER = ['SetC (in-network)', 'SetC→SetB', 'SetC→SetD']
TARGET_SET = {'SetC (in-network)': 'SetC', 'SetC→SetB': 'SetB', 'SetC→SetD': 'SetD'}

records = []
for fam in FAMILIES:
    for target in TARGET_ORDER:
        row = {'Family': fam, 'Target': target}
        row.update(_dep_row(fam, TARGET_SET[target]))
        records.append(row)

idx_cols = ['Family', 'Target']
t_cs_deploy = pd.DataFrame(records)[idx_cols + METRIC_COLS]
t_cs_deploy[METRIC_COLS] = t_cs_deploy[METRIC_COLS].round(4)

t_cs_deploy.to_csv(TAB_DIR / 'q6_appendix_cs_deploy.csv', index=False)
caption = ('Q6 appendix, Case Study B: means over runs of Accuracy, Precision, Recall, '
           'and Macro-F1 by model family (sequence-based, time split) for deployment '
           'from the strongest source SetC to SetC (in-network), SetB, and SetD. '
           'SetA is excluded as a target (same physical network as SetC).')
latex = t_cs_deploy.to_latex(index=False, float_format='%.4f', escape=False,
                             caption=caption, label='tab:q6_appendix_cs_deploy')
(TAB_DIR / 'q6_appendix_cs_deploy.tex').write_text(latex)
print('Saved q6_appendix_cs_deploy  shape:', t_cs_deploy.shape)
t_cs_deploy


Saved q6_appendix_cs_deploy  shape: (9, 6)


,Family,Target,Accuracy,Precision,Recall,Macro-F1
0,Deep Learning,SetC (in-network),0.9891,0.9397,0.7570,0.7996
1,Deep Learning,SetC→SetB,0.9332,0.8198,0.7604,0.7447
2,Deep Learning,SetC→SetD,0.7868,0.8025,0.6215,0.5977
3,Classical,SetC (in-network),0.9830,0.8566,0.7108,0.7439
4,Classical,SetC→SetB,0.8824,0.6727,0.6884,0.6026
5,Classical,SetC→SetD,0.8894,0.8258,0.5838,0.6052
6,Ensemble,SetC (in-network),0.9903,0.9053,0.7986,0.8270
7,Ensemble,SetC→SetB,0.9943,0.9755,0.9004,0.9321
8,Ensemble,SetC→SetD,0.9356,0.8632,0.6680,0.6901
